# S5.M9-1 I2 + I3 在其余 14 个任务上的真实 SGB 复核

**目的**：方法筛选第五阶段第 1 步（[DR-M5-016](../../../registry/decision_records/DR-M5-016.yaml) · D-1 ~ D-6）。

报告 v1.5 的头号结论「I2 + I3 普遍有效（14 / 17）」主要来自明文代理。真实 SGB 上只做过 3 个任务（UCI、Orange 偏好、Orange 大版流失）的「优化版联邦 − 原版联邦」。

本步在其余 14 个任务上补齐：
- 原版联邦（隐语经典默认）与优化版联邦（S5.M1 / S5.M5 选定配置）各一次真实 SGB（seed 11）；
- 已有 S5.M2 存档的 4 个任务（Santander G3、FedAds、mock 重合 80、COIL2000）复用其优化版联邦打分，只新拟合原版联邦。

**做法**：
- 新拟合用主动方明文直方图版本（S5.M7 已证与原版逐位一致），正式拟合前做两项抽查：
  - 前 2 棵树原版与明文的测试打分必须逐位相同；
  - 用明文前 2 / 4 棵外推全部棵数，超过 7,000 秒的不做。
- 分段运行，每段约 2 小时，可续跑。
- 对比口径同 S5.M2 / S5.M8：测试客户簇 bootstrap 1,000 次，单次拟合，区间不含训练随机性；另在 14 个任务之间做 Holm。

**数据**：公开数据，属开发证据。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m9_sgb_i2i3.yaml"
m9 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg = yaml.safe_load((REPO_ROOT / m9["base_config"]).read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("sgb_i2i3_confirm.py", "sgb_large_confirm.py", "sgb_cost_screen.py", "method_screen_sgb.py",
                        "method_screen.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in [CONFIG_PATH, REPO_ROOT / m9["decision_record"]] + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m9["decision_record"], "| step:", m9["step_id"], "|", m9["purpose"])
print("seed:", m9["seed"], "（每个 SGB 拟合一次）| git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
SGB, LARGE, M9 = (pathlib.Path(os.environ[m9[k]]) for k in ("sgb_dir_env", "large_dir_env", "out_dir_env"))
launch, n_seg = {}, 0
for line in (M9 / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    if line.startswith("=="):
        n_seg += 1
    elif "sha256:" in line:
        path, digest = line.split(" sha256:")
        launch.setdefault(pathlib.Path(path).name, set()).add(digest)
print("启动指纹：%d 段运行，每段记录的文件与 sha256（路径只留文件名）：" % n_seg)
for name, digests in sorted(launch.items()):
    print("  %s %s" % (name, ", ".join("sha256:" + d[:HASH_PREFIX_LEN] for d in sorted(digests))))
for p in COMPONENTS + [CONFIG_PATH]:
    assert launch[p.name] == {hashlib.sha256(p.read_bytes()).hexdigest()}, p.name
print("各段运行时的组件与配置与当前仓库逐字节相同（决策记录在运行后追加了复核，sha256 以上一行为准）。")

modules/m5_modeling/configs/s5_m9_sgb_i2i3.yaml sha256:02b104852e75
registry/decision_records/DR-M5-016.yaml sha256:680cc5366c7a
modules/m5_modeling/components/sgb_i2i3_confirm.py sha256:7b2d6ed9a322
modules/m5_modeling/components/sgb_large_confirm.py sha256:dc8d6cf4e244
modules/m5_modeling/components/sgb_cost_screen.py sha256:cf6088aa95be
modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
决策记录: registry/decision_records/DR-M5-016.yaml | step: S5.M9 | 其余 14 个任务上原版联邦与优化版联邦的真实 SGB，与同配置代理之差、SGB 上的 I2 + I3（优化版 − 原版）
seed: 11 （每个 SGB 拟合一次）| git: 4451a5b（工作区含本步未提交改动）
启动指纹：8 段运行，每段记录的文件与 sha256（路径只留文件名）：
  DR-M5-016.yaml sha256:e7dc7105091d
  method_screen.py sha256:05e19e025d0b
  method_screen_sgb.py sha256:2e54c618305f
  s5_m9_sgb_i2i3.yaml sha256:02b104852e75
  sgb_cost_screen.py sha256:cf6088aa95be
  sgb_i2i3_confirm.py sha256:7b2d6ed9a322
  sgb_large_confirm.py sha256:dc8d6cf4e244
各段运行时的组件与配置与当前仓库逐字节

## 1. 运行概况

下一格列出每个（任务，方法）的拟合信息：
- 来源：本步拟合，或复用 S5.M2 存档；
- 深度、棵数；
- 前 2 棵逐位抽查的结果；
- 外推秒数与实际拟合秒数。

In [2]:
from modules.m5_modeling.components import sgb_i2i3_confirm as IC

P = IC.MS.Params(cfg)
KEY = "%s − %s" % IC.PAIR
summary = json.loads((M9 / "summary.json").read_text(encoding="utf-8"))
EVAL = {t["name"]: json.loads((M9 / ("%s__eval.json" % t["name"])).read_text(encoding="utf-8")) for t in m9["tasks"]
        if (M9 / ("%s__eval.json" % t["name"])).exists()}
rows = []
for t in m9["tasks"]:
    n = t["name"]
    for m in IC.METHODS:
        pj, fj = M9 / ("%s__%s__probe.json" % (n, m)), M9 / ("%s__%s.json" % (n, m))
        pr = json.loads(pj.read_text(encoding="utf-8")) if pj.exists() else {}
        fi = json.loads(fj.read_text(encoding="utf-8")) if fj.exists() else {}
        rows.append({"任务": n, "方法": m, "来源": "S5.M2 存档" if (m == "F_opt_p" and t.get("reuse_opt")) else "本步拟合",
                     "深度": pr.get("depth", fi.get("depth")), "棵数": pr.get("rounds", fi.get("trees")),
                     "前 2 棵逐位一致": pr.get("identical_first_trees"), "外推秒数": pr.get("projected_fit_seconds"),
                     "实际秒数": fi.get("fit_seconds"), "判定": pr.get("verdict", "—")})
plan = pd.DataFrame(rows)
print("完成对比的任务：%d / %d；判定不做：%s" % (len(EVAL), len(m9["tasks"]), summary["skipped"] or "无"))
print(plan.to_string(index=False))

完成对比的任务：14 / 14；判定不做：无
                    任务      方法       来源  深度    棵数 前 2 棵逐位一致        外推秒数   实际秒数 判定
              coil2000   F_def     本步拟合 5.0 100.0      True  391.659497  361.8 ok
              coil2000 F_opt_p S5.M2 存档 NaN   NaN      None         NaN    NaN  —
        mock_overlap20   F_def     本步拟合 5.0 100.0      True  299.152758  272.4 ok
        mock_overlap20 F_opt_p     本步拟合 5.0  52.0      True  273.036360  256.1 ok
          orange_churn   F_def     本步拟合 5.0 100.0      True 1058.862463 1042.0 ok
          orange_churn F_opt_p     本步拟合 4.0  57.0      True  755.837848  750.1 ok
         orange_upsell   F_def     本步拟合 5.0 100.0      True 1124.908321 1064.4 ok
         orange_upsell F_opt_p     本步拟合 6.0  51.0      True 1363.190612 1444.9 ok
        mock_overlap50   F_def     本步拟合 5.0 100.0      True  458.040277  409.1 ok
        mock_overlap50 F_opt_p     本步拟合 6.0  62.0      True  671.518529  649.9 ok
          santander_tx   F_def     本步拟合 5.0 100.0      True 1626.309059 155

## 2. 复现

完整拟合每个（任务，方法）约 5 分钟到 75 分钟，notebook 不重拟合 SGB。下一格在第一个任务（COIL2000）上做两件事：
- 用仓库组件重拟合同配置代理，从存档打分重算 AUROC 与 bootstrap 对比；
- 重算跨任务汇总与 Holm。

要求与批量运行相同。

In [3]:
REPRO_TOL = 1e-12
NAME = m9["tasks"][0]["name"]
res = json.loads((RUN / (NAME + ".json")).read_text(encoding="utf-8"))
sc = IC.MS.Screen(NAME, cfg, PRIV, M9 / "unused")
again = IC.evaluate_task(P, sc, res, {m: np.load(M9 / ("%s__%s.npy" % (NAME, m))).astype(float) for m in IC.METHODS})
for m in IC.METHODS:
    a, b = again["methods"][m]["sgb_minus_proxy"], EVAL[NAME]["methods"][m]["sgb_minus_proxy"]
    assert all(abs(a[k] - b[k]) <= REPRO_TOL for k in ("diff", "ci_low", "ci_high", "p")), m
c1, c2 = again["contrasts"][KEY]["sgb"], EVAL[NAME]["contrasts"][KEY]["sgb"]
assert all(abs(c1[k] - c2[k]) <= REPRO_TOL for k in ("diff", "ci_low", "ci_high", "p"))
again_sum = IC.summarize(EVAL, summary["skipped"], m9["proxy_tol"])
assert all(abs(a["sgb_p_holm"] - b["sgb_p_holm"]) <= REPRO_TOL for a, b in zip(again_sum["tasks"], summary["tasks"]))
print("%s：用仓库组件重拟合同配置代理、从存档打分重算 AUROC 与 bootstrap 对比，与批量运行相同（差 ≤ %g）；跨任务汇总与 Holm 重算相同。" % (NAME, REPRO_TOL))

coil2000：用仓库组件重拟合同配置代理、从存档打分重算 AUROC 与 bootstrap 对比，与批量运行相同（差 ≤ 1e-12）；跨任务汇总与 Holm 重算相同。


## 3. 结果

- **SGB 与同配置代理**：两者之差的配对区间，以及差的绝对值是否在 0.005 以内（DR-M5-012 · D-1）。
- **SGB 上的 I2 + I3**（优化版 − 原版）：配对差、区间、p 与 14 个任务内的 Holm，与 S5.M1 / S5.M5 代理（6 seed）的结论逐任务对照。
- **合并已提交的 3 个任务**：只读已公开的汇总 CSV，得到 17 个任务的完整 SGB 证据。
- **只作描述**：联邦 SGB − 内地单边代理。
- 汇总 CSV 写入 `results/sgb_i2i3_v1/`。

In [4]:
OUT = REPO_ROOT / m9["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
R, TOL = m9["round_auc"], m9["proxy_tol"]
vs = pd.DataFrame([{"dataset": n, "method": m, "sgb_auroc": v["sgb_auroc"], "proxy_auroc": v["proxy_auroc"],
                    "sgb_minus_proxy": v["sgb_minus_proxy"]["diff"], "ci_low": v["sgb_minus_proxy"]["ci_low"],
                    "ci_high": v["sgb_minus_proxy"]["ci_high"], "within_tolerance": abs(v["sgb_minus_proxy"]["diff"]) <= TOL,
                    "source": "S5.M2 存档" if (m == "F_opt_p" and ev.get("reuse_opt")) else "S5.M9"}
                   for n, ev in EVAL.items() for m, v in ev["methods"].items()])
con = pd.DataFrame([{"dataset": r["task"], "contrast": KEY, "sgb_diff": r["sgb_diff"], "sgb_ci_low": r["sgb_ci_low"],
                     "sgb_ci_high": r["sgb_ci_high"], "sgb_p": r["sgb_p"], "sgb_p_holm_14": r["sgb_p_holm"],
                     "proxy_6seed_diff": r["proxy_diff"], "proxy_6seed_ci_low": EVAL[r["task"]]["contrasts"][KEY]["proxy_6seed"]["ci_low"],
                     "proxy_6seed_ci_high": EVAL[r["task"]]["contrasts"][KEY]["proxy_6seed"]["ci_high"],
                     "sgb_holds": r["sgb_holds"], "proxy_holds": r["proxy_holds"], "agree": r["agree"]} for r in summary["tasks"]])
desc = pd.DataFrame([{"dataset": n, "method": m, **v} for n, ev in EVAL.items() for m, v in ev["descriptive"].items()])
probes = plan.rename(columns={"任务": "dataset", "方法": "method", "来源": "source", "深度": "depth", "棵数": "trees",
                              "前 2 棵逐位一致": "identical_first_two_trees", "外推秒数": "projected_seconds",
                              "实际秒数": "fit_seconds", "判定": "verdict"})
# 与已提交的 3 个任务（S5.M2：UCI、Orange 偏好；S5.M8：Orange 大版流失）合并成 17 个任务的 SGB 证据（只读已公开的汇总 CSV）
prev = pd.concat([pd.read_csv(REPO_ROOT / "modules/m5_modeling/results/method_screen_sgb_v1/sgb_contrasts.csv"),
                  pd.read_csv(REPO_ROOT / "modules/m5_modeling/results/sgb_large_v1/sgb_contrasts.csv")])
prev = prev[prev["contrast"] == KEY].assign(source="已提交（S5.M2 / S5.M8）")
all17 = pd.concat([con.assign(source="S5.M9")[prev.columns.tolist()], prev], ignore_index=True)
all17["sgb_holds"] = all17["sgb_ci_low"] > 0
all17["proxy_holds"] = all17["proxy_6seed_ci_low"] > 0
vs.to_csv(OUT / "sgb_vs_proxy.csv", index=False, float_format="%.5f")
con.to_csv(OUT / "sgb_i2i3_contrasts.csv", index=False, float_format="%.5f")
desc.to_csv(OUT / "sgb_vs_guest_only_descriptive.csv", index=False, float_format="%.5f")
probes.to_csv(OUT / "probes_and_fits.csv", index=False, float_format="%.1f")
all17.to_csv(OUT / "i2i3_sgb_all17.csv", index=False, float_format="%.5f")
print("SGB 与同配置代理（容差 %.3f）：" % TOL)
print(vs.round(R).to_string(index=False))
print("超出容差的（任务，方法）：%d / %d" % ((~vs["within_tolerance"]).sum(), len(vs)))
print("SGB 上的 I2 + I3（优化版 − 原版），14 个任务内 Holm：")
print(con.round(R).to_string(index=False))
print("I2 + I3 在 SGB 上成立 %d / %d；代理（6 seed）成立 %d / %d；两者结论一致 %d / %d" % (
    summary["sgb_holds"], summary["n"], summary["proxy_holds"], summary["n"], summary["agree"], summary["n"]))
print("合并已提交的 3 个任务（共 %d 个）：SGB 上成立 %d，代理成立 %d" % (len(all17), all17["sgb_holds"].sum(), all17["proxy_holds"].sum()))
print("只作描述：联邦 SGB − 内地单边代理（6 seed）：")
print(desc.round(R).to_string(index=False))

SGB 与同配置代理（容差 0.005）：
               dataset  method  sgb_auroc  proxy_auroc  sgb_minus_proxy  ci_low  ci_high  within_tolerance   source
              coil2000   F_def     0.6828       0.6967          -0.0139 -0.0309   0.0043             False    S5.M9
              coil2000 F_opt_p     0.7331       0.7275           0.0056  0.0020   0.0094             False S5.M2 存档
        mock_overlap20   F_def     0.8932       0.8928           0.0004 -0.0122   0.0125              True    S5.M9
        mock_overlap20 F_opt_p     0.9069       0.9026           0.0043 -0.0078   0.0154              True    S5.M9
          orange_churn   F_def     0.7003       0.7099          -0.0097 -0.0236   0.0043             False    S5.M9
          orange_churn F_opt_p     0.7311       0.7305           0.0006 -0.0045   0.0054              True    S5.M9
         orange_upsell   F_def     0.8335       0.8341          -0.0006 -0.0101   0.0091              True    S5.M9
         orange_upsell F_opt_p     0.8667       0.

## 4. 结论

1. **明文抽查与时长**：
   - 24 次新拟合的前 2 棵，原版与明文打分全部逐位相同；
   - 外推与实际拟合秒数之比在 0.86 ~ 1.09 之间，没有拟合超出上限；
   - 分 8 段完成，合计拟合约 10.3 小时。
2. **I2 + I3 在真实 SGB 上：14 个任务中 12 个成立**（区间不含 0）。
   - 其中 9 个经 14 个任务内的 Holm 后仍成立；
   - 不成立的 2 个：mock 重合 50 +0.0042 [−0.0034, +0.0114]、IEEE 第 1 块 +0.0096 [−0.0025, +0.0229]；
   - 合并已提交的 3 个任务，**17 个任务中 15 个在真实 SGB 上成立**。
3. **与代理结论的对照：14 个中 11 个一致**，不一致的 3 个：
   - mock 重合 20（+0.0137）与 mock 重合 80（+0.0075）：SGB 上成立，代理上不成立（+0.008、+0.002）；
   - IEEE 第 1 块：代理上成立（+0.016），SGB 上区间含 0。
   - 多个任务上 SGB 的幅度大于代理，如 COIL2000 +0.050 对 +0.031、Orange 流失 +0.031 对 +0.021、HomeCredit +0.021 对 +0.015。
4. **SGB 与同配置代理之差：28 个配置中 7 个超出 0.005**，按 DR-M5-012 · D-1 把这些配置的代理结论降级为「待 SGB 确认」。
   - 原版联邦 5 个：COIL2000 −0.014、Orange 流失 −0.010、Orange 大版追加销售 −0.006、HomeCredit −0.005，IEEE 第 1 块 +0.009；
   - 优化版联邦 2 个：COIL2000 +0.006、mock 重合 80 +0.006（S5.M2 已知）。
   - 原版参数下超差的方向不一致：本步多数是 SGB 低于代理，S5.M2 的 UCI 与 Orange 偏好是 SGB 高于代理。所以「原版 → 优化版」的幅度只能以 SGB 为准。
5. **只作描述**：优化版联邦 SGB 比内地单边代理高 −0.001 ~ +0.074，Orange 升级上仍看不出联邦增益（−0.001），与 S5.M1 一致。

**对报告的意义**：
- 「I2 + I3 普遍有效」现在有 17 个任务的完整真实 SGB 证据：15 / 17 成立。
- 代理对原版参数的估计误差比 S5.M2 看到的大，方向也不一致；报告里原版参数相关的幅度应以 SGB 为准。

**边界**：
- 每个（任务，方法）单次拟合（seed 11），区间只含测试集抽样，不含训练随机性；
- 明文直方图只在实验代码里替换一个方法，用于正式联调需经安全评审；完整拟合的逐位一致只在 S5.M7 的 6 个任务上证实，其余任务是前 2 棵抽查；
- 公开数据上的开发证据，不换算成汇丰的命中率或收益；技术侧自评，未经法务复核（W-001）。